Que.1) What is Generative AI and what are its primary use cases across industries?
Ans:- Generative AI refers to a class of artificial intelligence systems designed to create new content—whether text, images, audio, video, or even code—by learning patterns from existing data. Unlike traditional AI, which focuses on classification or prediction, generative AI emphasizes production of novel outputs that resemble human creativity.
Generative AI models (like GPT, DALL·E, or Stable Diffusion) are trained on large datasets and use deep learning techniques—often involving neural networks such as transformers or variational autoencoders—to generate outputs that are coherent, contextually relevant, and stylistically aligned with the training data.

Que.2) Explain the role of probabilistic modeling in generative models. How do these models differ from discriminative models?
Ans:- 
- Generative models aim to model the joint probability distribution 𝑃(𝑋,𝑌) of inputs 𝑋 and outputs/labels 𝑌.
- Probabilistic modeling allows these models to:
    - Capture uncertainty in data and generate diverse outputs.
    - Sample new data points by drawing from the learned probability distribution.
    - Estimate likelihoods of observed data, which is useful for anomaly detection and density estimation.
- Example: In a language model, probabilistic modeling defines the likelihood of the next word given previous words, enabling coherent text generation.

Que.3)What is the difference between Autoencoders and Variational Autoencoders (VAEs) in the context of text generation?
Ans:- 
a) Autoencoder in text generation:-
- An Autoencoder is a neural network that learns to compress input data into a lower-dimensional representation (encoder) and then reconstruct it back (decoder).
- In text generation:
    - Autoencoders capture latent features of text (semantic meaning, structure).
    - They are mainly used for dimensionality reduction, denoising, and representation learning.
    - Limitation: Standard autoencoders do not explicitly model probability distributions, so they struggle with generating diverse and novel text samples.

b)Variational Autoencoder (VAE) in text generation:-
- A VAE extends the autoencoder by introducing probabilistic modeling.
- Instead of encoding input into a fixed latent vector, VAEs encode it into a distribution (mean and variance).
- During generation, new samples can be drawn from this distribution, enabling stochastic and diverse text outputs.
- In text generation:
   - VAEs can generate sentences with variability and creativity.
   - They are often used in dialogue systems, creative writing, and paraphrasing tasks.

Que.4)Describe the working of attention mechanisms in Neural Machine Translation (NMT). Why are they critical? 
Ans:-
Working of attention mechanisms in NMT:-In traditional sequence-to-sequence NMT models, the encoder compresses the entire source sentence into a single fixed-length vector. This often leads to information loss, especially for long sentences. Attention mechanism solves this by allowing the decoder to dynamically focus on different parts of the source sentence at each decoding step.
Working:
1. Encoder produces hidden states for each input token.
2. Attention layer computes a set of weights (called attention scores) that measure the relevance of each source token to the current target token being generated.
3. These weights are applied to the encoder hidden states to form a context vector.
4. The decoder uses this context vector along with its own hidden state to predict the next word in the translation.

attention mechanisms are critical in NMT. AS->
- Handle long sentences → Avoids bottleneck of fixed-length vectors by letting the model access all source words.
- Improves translation quality → Captures word alignments between source and target languages.
- Provides interpretability → Attention weights show which source words influenced each translated word.
- Enables context-aware translation → Especially important for languages with flexible word order.
- Foundation for Transformers → Attention mechanisms evolved into self-attention, the core of modern Transformer-based models (e.g., BERT, GPT, T5).

Que.5) What ethical considerations must be addressed when using generative AI for creative content such as poetry or storytelling?
Ans:-Generative AI introduces several ethical challenges in creative domains:
1. Authorship & Ownership
- Who owns AI-generated content? The user, the developer, or the AI system?
- Lack of clarity can lead to disputes over intellectual property rights.
2. Originality & Plagiarism
- AI may unintentionally reproduce phrases or styles from training data.
- Risk of hidden plagiarism if outputs closely mimic copyrighted works.
3. Bias & Representation
- Training data may contain cultural, gender, or racial biases.
- AI-generated stories could reinforce stereotypes or misrepresent communities.
4. Authenticity & Transparency
- Readers may assume content is human-authored.
- Ethical use requires disclosure when AI is involved in creation.
5. Cultural Sensitivity
- AI may generate content that misuses cultural symbols, traditions, or languages.
- This can lead to appropriation or disrespect.
6. Emotional Manipulation
- Creative writing influences emotions; misuse could spread propaganda or misinformation disguised as art.

Example in Poetry/Storytelling Context:-
- If AI generates a poem about grief, it must avoid trivializing sensitive emotions.
- If AI writes a story inspired by indigenous folklore, it should respect cultural boundaries and not misappropriate sacred narratives.

In [1]:
# Ques.6)Use the following small text dataset to train a simple Variational 
# Autoencoder (VAE) for text reconstruction: 
# ["The sky is blue", "The sun is bright", "The grass is green",  
# "The night is dark", "The stars are shining"] 
# 1. Preprocess the data (tokenize and pad the sequences). 
# 2. Build a basic VAE model for text reconstruction. 
# 3. Train the model and show how it reconstructs or generates similar sentences.

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# -----------------------------
# 1. Preprocess the dataset
# -----------------------------
texts = ["The sky is blue", "The sun is bright",
         "The grass is green", "The night is dark",
         "The stars are shining"]

# Build vocabulary
all_words = set(" ".join(texts).split())
word2idx = {w:i+1 for i,w in enumerate(all_words)}  # start at 1
idx2word = {i:w for w,i in word2idx.items()}

max_len = max(len(t.split()) for t in texts)

# Encode and pad
encoded = []
for t in texts:
    seq = [word2idx[w] for w in t.split()]
    seq += [0]*(max_len-len(seq))  # pad with 0
    encoded.append(seq)

data = torch.tensor(encoded)
dataset = TensorDataset(data)
loader = DataLoader(dataset, batch_size=2, shuffle=True)

# -----------------------------
# 2. Build a simple VAE model
# -----------------------------
class VAE(nn.Module):
    def __init__(self, vocab_size, embed_dim=16, hidden_dim=32, latent_dim=8):
        super(VAE, self).__init__()
        self.embedding = nn.Embedding(vocab_size+1, embed_dim, padding_idx=0)
        self.encoder = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.z_mean = nn.Linear(hidden_dim, latent_dim)
        self.z_logvar = nn.Linear(hidden_dim, latent_dim)
        self.decoder = nn.LSTM(latent_dim, hidden_dim, batch_first=True)
        self.out = nn.Linear(hidden_dim, vocab_size+1)

    def forward(self, x):
        emb = self.embedding(x)
        _, (h, _) = self.encoder(emb)
        h = h[-1]
        mean = self.z_mean(h)
        logvar = self.z_logvar(h)
        std = torch.exp(0.5*logvar)
        eps = torch.randn_like(std)
        z = mean + eps*std
        z = z.unsqueeze(1).repeat(1, x.size(1), 1)  # repeat for sequence length
        dec_out, _ = self.decoder(z)
        return self.out(dec_out), mean, logvar

# -----------------------------
# 3. Train the model
# -----------------------------
vae = VAE(vocab_size=len(word2idx))
optimizer = optim.Adam(vae.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss(ignore_index=0)

for epoch in range(50):
    for batch in loader:
        x = batch[0]
        optimizer.zero_grad()
        preds, mean, logvar = vae(x)
        # reconstruction loss
        recon_loss = criterion(preds.view(-1, preds.size(-1)), x.view(-1))
        # KL divergence
        kl_loss = -0.5 * torch.mean(1 + logvar - mean.pow(2) - logvar.exp())
        loss = recon_loss + kl_loss
        loss.backward()
        optimizer.step()
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

# -----------------------------
# 4. Reconstruct / generate
# -----------------------------
vae.eval()
with torch.no_grad():
    preds, _, _ = vae(data)
    decoded = torch.argmax(preds, dim=-1).numpy()
    for seq in decoded:
        sentence = " ".join(idx2word.get(idx, "") for idx in seq if idx != 0)
        print("Generated:", sentence)


Epoch 0, Loss: 2.6337
Epoch 10, Loss: 2.1148
Epoch 20, Loss: 1.3331
Epoch 30, Loss: 0.8678
Epoch 40, Loss: 1.3534
Generated: The sky is blue
Generated: The sun is bright
Generated: The night is dark
Generated: The grass is green
Generated: The stars are shining


In [5]:
#Que.7) Use a pre-trained GPT model (like GPT-2 or GPT-3) to translate a short English paragraph into French and German. Provide the original and translated text.

# Step 1: Install required libraries
#!pip install transformers torch sentencepiece

# Step 2: Import libraries
from transformers import MarianMTModel, MarianTokenizer

# Step 3: Define a helper function for translation
def translate(text, model_name):
    tokenizer = MarianTokenizer.from_pretrained(model_name)
    model = MarianMTModel.from_pretrained(model_name)
    inputs = tokenizer(text, return_tensors="pt", padding=True)
    translated = model.generate(**inputs, max_length=100)
    return tokenizer.decode(translated[0], skip_special_tokens=True)

# Step 4: Original English text
text = ("Generative AI is transforming industries by enabling machines "
        "to create human-like content. It is used in healthcare, education, "
        "and entertainment to accelerate innovation and personalize experiences.")

# Step 5: Translate into French and German
translation_fr = translate(text, "Helsinki-NLP/opus-mt-en-fr")
translation_de = translate(text, "Helsinki-NLP/opus-mt-en-de")

# Step 6: Print results
print("Original English Text:\n", text)
print("\nFrench Translation:\n", translation_fr)
print("\nGerman Translation:\n", translation_de)


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Original English Text:
 Generative AI is transforming industries by enabling machines to create human-like content. It is used in healthcare, education, and entertainment to accelerate innovation and personalize experiences.

French Translation:
 L'IA génératrice transforme les industries en permettant aux machines de créer des contenus ressemblant à des êtres humains. Elle est utilisée dans les soins de santé, l'éducation et le divertissement pour accélérer l'innovation et personnaliser les expériences.

German Translation:
 Generative KI transformiert Industrien, indem sie Maschinen ermöglicht, menschliche Inhalte zu schaffen. Es wird in der Gesundheitsversorgung, Bildung und Unterhaltung verwendet, um Innovation zu beschleunigen und Erfahrungen zu personalisieren.


In [6]:
#Que.8) : Implement a simple attention-based encoder-decoder model for English-to-Spanish translation using Tensorflow or PyTorch.
# !pip install torch

import torch, torch.nn as nn, torch.optim as optim

# Toy dataset
pairs = [("hello","hola"),("how are you","como estas"),
         ("good morning","buenos dias"),("thank you","gracias"),
         ("good night","buenas noches")]

# Build vocab
eng, spa = {"<pad>":0,"<sos>":1,"<eos>":2}, {"<pad>":0,"<sos>":1,"<eos>":2}
for e,s in pairs:
    for w in e.split(): eng.setdefault(w,len(eng))
    for w in s.split(): spa.setdefault(w,len(spa))
inv_spa = {v:k for k,v in spa.items()}

def encode(seq,v): return [v["<sos>"]]+[v[w] for w in seq.split()]+[v["<eos>"]]
data=[(encode(e,eng),encode(s,spa)) for e,s in pairs]
max_e,max_s=max(len(e) for e,_ in data),max(len(s) for _,s in data)
pad=lambda x,m:x+[0]*(m-len(x))
X=torch.tensor([pad(e,max_e) for e,_ in data]); Y=torch.tensor([pad(s,max_s) for _,s in data])

# Encoder, Attention, Decoder
class Enc(nn.Module):
    def __init__(self,vin,emb,hid): super().__init__()
    self.emb=nn.Embedding(vin,emb); self.rnn=nn.GRU(emb,hid,batch_first=True)
    def forward(self,x): return self.rnn(self.emb(x))
class Attn(nn.Module):
    def __init__(self,hid): super().__init__()
    self.fc=nn.Linear(hid*2,hid); self.v=nn.Linear(hid,1,bias=False)
    def forward(self,h,enc):
        h=h[-1].unsqueeze(1).repeat(1,enc.size(1),1)
        e=torch.tanh(self.fc(torch.cat((h,enc),2)))
        w=torch.softmax(self.v(e).squeeze(2),1)
        return torch.bmm(w.unsqueeze(1),enc)
class Dec(nn.Module):
    def __init__(self,vout,emb,hid): super().__init__()
    self.emb=nn.Embedding(vout,emb); self.rnn=nn.GRU(emb+hid,hid,batch_first=True)
    self.fc=nn.Linear(hid*2,vout); self.att=Attn(hid)
    def forward(self,x,h,enc):
        e=self.emb(x).unsqueeze(1); c=self.att(h,enc)
        o,h=self.rnn(torch.cat((e,c),2),h)
        return self.fc(torch.cat((o.squeeze(1),c.squeeze(1)),1)),h

enc,dec=Enc(len(eng),32,64),Dec(len(spa),32,64)
opt=optim.Adam(list(enc.parameters())+list(dec.parameters()),lr=0.01)
loss_fn=nn.CrossEntropyLoss(ignore_index=0)

# Train
for ep in range(150):
    tot=0
    for i in range(len(X)):
        src,trg=X[i].unsqueeze(0),Y[i].unsqueeze(0)
        enc_out,h=enc(src); dh=h; loss=0
        for t in range(trg.size(1)-1):
            out,dh=dec(trg[:,t],dh,enc_out)
            loss+=loss_fn(out,trg[:,t+1])
        opt.zero_grad(); loss.backward(); opt.step(); tot+=loss.item()
    if ep%50==0: print(f"Epoch {ep}, Loss {tot:.2f}")

# Translate
def translate(sent):
    src=torch.tensor(pad(encode(sent,eng),max_e)).unsqueeze(0)
    enc_out,h=enc(src); dh=h; x=torch.tensor([spa["<sos>"]]); res=[]
    for _ in range(max_s):
        out,dh=dec(x,dh,enc_out); p=out.argmax(1).item()
        if p==spa["<eos>"]: break
        res.append(inv_spa[p]); x=torch.tensor([p])
    return " ".join(res)

print("\nTest:")
print("English: hello -> Spanish:",translate("hello"))
print("English: good night -> Spanish:",translate("good night"))


NameError: name 'vin' is not defined

In [4]:
# Que.9) Use the following short poetry dataset to simulate poem generation with a 
# pre-trained GPT model: 
# ["Roses are red, violets are blue,", 
# "Sugar is sweet, and so are you.", 
# "The moon glows bright in silent skies,", 
# "A bird sings where the soft wind sighs."] 
# Using this dataset as a reference for poetic structure and language, generate a new 2-4 
# line poem using a pre-trained GPT model (such as GPT-2). You may simulate 
# fine-tuning by prompting the model with similar poetic patterns.

# Step 1: Install required libraries
#!pip install transformers torch

# Step 2: Import libraries
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch

# Step 3: Load pre-trained GPT-2
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

# Step 4: Poetry dataset (used as prompt for style)
poetry_dataset = [
    "Roses are red, violets are blue,",
    "Sugar is sweet, and so are you.",
    "The moon glows bright in silent skies,",
    "A bird sings where the soft wind sighs."
]

# Combine dataset into a single prompt
prompt = "\n".join(poetry_dataset) + "\n"

# Step 5: Encode and generate continuation
inputs = tokenizer.encode(prompt, return_tensors="pt")
outputs = model.generate(
    inputs,
    max_length=60,       # generate 2–4 extra lines
    num_return_sequences=1,
    temperature=0.8,     # creativity
    top_p=0.9,           # nucleus sampling
    do_sample=True
)

# Step 6: Decode and print
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("=== Original Prompt ===")
print(prompt)
print("\n=== Generated Poem ===")
# Only show the continuation after the dataset
print(generated_text[len(prompt):].strip())


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

=== Original Prompt ===
Roses are red, violets are blue,
Sugar is sweet, and so are you.
The moon glows bright in silent skies,
A bird sings where the soft wind sighs.


=== Generated Poem ===
The wind brings a strong wind,
And a man brings a strong wind.


Que.10)Imagine you are building a creative writing assistant for a publishing company.
The assistant should generate story plots and character descriptions using Generative AI.
Describe how you would design the system, including model selection, training data, bias mitigation, and evaluation methods.
Explain the real-world challenges you might face.

Ans:- 
Designing a Creative Writing Assistant with Generative AI:-
1. Model Selection
- Use a large language model such as GPT‑3.5 or GPT‑4 for creative text generation.
- Apply prompt engineering or lightweight fine‑tuning to adapt the model for story plots and character descriptions.
- Combine with smaller classifiers (e.g., genre or sentiment models) to guide tone and style.

2. Training Data
- Curate a diverse dataset of published story synopses, character sketches, and plot outlines across multiple genres.
- Incorporate anonymized internal manuscripts (with author consent) to align with the company’s editorial style.
- Include style guides and narrative templates to ensure outputs match publishing standards.

3. Bias Mitigation
- Balance datasets to represent different cultures, genders, and perspectives, reducing stereotypical outputs.
- Use toxicity and bias detection filters to screen generated text.
- Keep a human‑in‑the‑loop: editors review and refine AI‑generated plots before use.

4. Evaluation Methods
- Automatic metrics: BLEU/ROUGE for overlap, diversity scores for originality.
- Human evaluation: Editors rate creativity, coherence, and alignment with publishing goals.
- A/B testing: Compare AI‑assisted plots against human baselines in pilot projects.

5. Real‑World Challenges
- Creativity vs. cliché: Models may recycle tropes instead of producing fresh ideas.
- Bias and representation: Risk of reinforcing stereotypes if training data isn’t diverse.
- Intellectual property: Ensuring outputs don’t plagiarize existing works.
- Trust and adoption: Convincing editors that AI is a collaborator, not a replacement.
- Cost and scalability: Managing compute resources while supporting iterative creative workflows.